# Notebook 01: Transcripcion

Transcribe grabaciones MP3 de sesiones de clase usando **mlx-whisper** (Apple Silicon Neural Engine) con **large-v3-turbo**.

**Output:** `artifacts/llm_{id}/audio.json` por sesion (compatible con el pipeline RAG).

In [ ]:
import json
import time
from pathlib import Path
from datetime import timedelta

PROJECT_ROOT = Path('/Users/alexmartin/Documents/video')
OUTPUTS_DIR = PROJECT_ROOT / 'outputs' / 'llm_sessions'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Buscar TODOS los MP3 en recordings/ (excluir < 1MB = false starts)
MP3_FILES = sorted(PROJECT_ROOT.glob('recordings/*.mp3'))
MP3_FILES = [f for f in MP3_FILES if f.stat().st_size > 1_000_000]

# Filtrar: solo los que NO tienen audio.json ya generado
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
pending = []
skipped = []
for f in MP3_FILES:
    session_num = f.stem.split('_')[0]
    out_file = ARTIFACTS_DIR / f'llm_{session_num}' / 'audio.json'
    if out_file.exists():
        skipped.append(f'  [skip] {session_num}: {f.name} (ya existe)')
    else:
        pending.append(f)

print(f'MP3 encontrados: {len(MP3_FILES)} | Pendientes: {len(pending)} | Ya procesados: {len(skipped)}')
if skipped:
    print(f'\nYa procesados:')
    for s in skipped:
        print(s)
if pending:
    print(f'\nPendientes de transcribir:')
    for f in pending:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name} ({size_mb:.0f} MB)')
else:
    print('\nTodo está al día, no hay nada nuevo que transcribir.')

MP3_FILES = pending  # Solo procesar los pendientes

In [ ]:
import mlx_whisper

# large-v3-turbo: mejor calidad + rápido en Apple Silicon
MODEL_REPO = 'mlx-community/whisper-large-v3-turbo'

print(f'Modelo: {MODEL_REPO}')
print('(se descarga automáticamente la primera vez, ~1.5GB)')

In [ ]:
# Transcribir cada sesión y guardar individualmente
all_sessions = []

for i, mp3_path in enumerate(MP3_FILES):
    session_name = mp3_path.stem
    session_num = session_name.split('_')[0]  # e.g. '38'
    output_file = OUTPUTS_DIR / f'{session_name}_whisper.json'
    
    # Skip si ya existe
    if output_file.exists():
        print(f'[{i+1}/{len(MP3_FILES)}] {session_num} ya existe, cargando...')
        with open(output_file) as f:
            session_data = json.load(f)
        all_sessions.append(session_data)
        continue
    
    print(f'\n[{i+1}/{len(MP3_FILES)}] Transcribiendo sesión {session_num}: {mp3_path.name}')
    t0 = time.time()
    
    result = mlx_whisper.transcribe(
        str(mp3_path),
        path_or_hf_repo=MODEL_REPO,
        language='es',
    )
    
    elapsed = time.time() - t0
    
    whisper_segments = []
    for seg in result['segments']:
        whisper_segments.append({
            'start_ms': int(seg['start'] * 1000),
            'end_ms': int(seg['end'] * 1000),
            'text': seg['text'].strip(),
        })
    
    duration_s = result.get('duration', whisper_segments[-1]['end_ms'] / 1000 if whisper_segments else 0)
    duration_min = duration_s / 60
    
    session_data = {
        'session': session_num,
        'file': mp3_path.name,
        'duration_min': round(duration_min, 1),
        'processing_time_min': round(elapsed / 60, 1),
        'realtime_factor': round(elapsed / duration_s, 2) if duration_s else None,
        'num_segments': len(whisper_segments),
        'model': MODEL_REPO,
        'language': 'es',
        'segments': whisper_segments,
    }
    
    # Guardar individual
    with open(output_file, 'w') as f:
        json.dump(session_data, f, indent=2, ensure_ascii=False)
    
    all_sessions.append(session_data)
    
    print(f'  Duración: {duration_min:.0f} min | Procesado en: {elapsed/60:.1f} min (x{session_data["realtime_factor"]})')
    print(f'  Segmentos: {len(whisper_segments)} | Guardado: {output_file.name}')

print(f'\nTodas las sesiones procesadas: {len(all_sessions)}')

In [ ]:
# Resumen de todas las sesiones
print(f'{"Sesión":>8} | {"Duración":>10} | {"Segmentos":>10} | {"Tiempo proc.":>12} | {"RTF":>5}')
print('-' * 60)

total_dur = 0
total_segs = 0
total_proc = 0

for s in all_sessions:
    dur = s['duration_min']
    total_dur += dur
    total_segs += s['num_segments']
    proc = s.get('processing_time_min', 0)
    total_proc += proc
    rtf = s.get('realtime_factor', '?')
    print(f'{s["session"]:>8} | {dur:>8.0f} min | {s["num_segments"]:>10} | {proc:>10.1f} min | x{rtf}')

print('-' * 60)
print(f'{"TOTAL":>8} | {total_dur:>8.0f} min | {total_segs:>10} | {total_proc:>10.1f} min |')
print(f'\n({total_dur/60:.1f} horas de clase)')

In [ ]:
# Guardar consolidado: todas las sesiones en un solo archivo
consolidated = {
    'course': 'Large Language Models',
    'sessions': list(range(38, 44)),
    'total_duration_min': round(total_dur, 1),
    'total_segments': total_segs,
    'data': all_sessions,
}

consolidated_path = OUTPUTS_DIR / 'all_sessions_whisper.json'
with open(consolidated_path, 'w') as f:
    json.dump(consolidated, f, indent=2, ensure_ascii=False)

print(f'Consolidado guardado: {consolidated_path}')
print(f'Tamaño: {consolidated_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Exportar transcripción como texto plano (para lectura rápida)
txt_path = OUTPUTS_DIR / 'transcripcion_completa.txt'

with open(txt_path, 'w') as f:
    for s in all_sessions:
        f.write(f"\n{'='*80}\n")
        f.write(f"SESIÓN {s['session']} - {s['file']} ({s['duration_min']:.0f} min)\n")
        f.write(f"{'='*80}\n\n")
        for seg in s['segments']:
            ts = str(timedelta(milliseconds=seg['start_ms']))[:-3]  # HH:MM:SS.mmm
            f.write(f"[{ts}] {seg['text']}\n")
        f.write('\n')

print(f'Transcripción texto plano: {txt_path}')
print(f'Tamaño: {txt_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Preview: primeros segmentos de cada sesión
for s in all_sessions:
    print(f"\n--- Sesión {s['session']} ({s['duration_min']:.0f} min, {s['num_segments']} segs) ---")
    for seg in s['segments'][:3]:
        ts = str(timedelta(milliseconds=seg['start_ms']))[:-3]
        print(f"  [{ts}] {seg['text'][:100]}")
    print('  ...')

---

## Generar artifacts para el pipeline RAG

Convierte la transcripcion al formato `artifacts/llm_{session}/audio.json` compatible con el resto del pipeline.
Computa ademas: silence spans (gaps entre segmentos), speech rate (WPS en ventanas), y energy (RMS desde el MP3).

In [ ]:
import numpy as np

def compute_silence_spans(segments, duration_ms, min_silence_ms=500):
    """Detecta silencios como gaps entre segmentos consecutivos."""
    silences = []
    for i in range(len(segments) - 1):
        gap_start = segments[i]['end_ms']
        gap_end = segments[i + 1]['start_ms']
        gap_ms = gap_end - gap_start
        if gap_ms >= min_silence_ms:
            silences.append({
                'start_ms': gap_start,
                'end_ms': gap_end,
                'duration_ms': gap_ms,
            })
    if segments and segments[0]['start_ms'] >= min_silence_ms:
        silences.insert(0, {
            'start_ms': 0,
            'end_ms': segments[0]['start_ms'],
            'duration_ms': segments[0]['start_ms'],
        })
    if segments and (duration_ms - segments[-1]['end_ms']) >= min_silence_ms:
        silences.append({
            'start_ms': segments[-1]['end_ms'],
            'end_ms': duration_ms,
            'duration_ms': duration_ms - segments[-1]['end_ms'],
        })
    return silences


def compute_wps_windows(segments, duration_ms, window_ms=5000):
    """Calcula words-per-second en ventanas deslizantes."""
    windows = []
    for t in range(0, duration_ms, window_ms):
        w_start = t
        w_end = t + window_ms
        word_count = 0
        for seg in segments:
            if seg['end_ms'] <= w_start or seg['start_ms'] >= w_end:
                continue
            overlap_start = max(seg['start_ms'], w_start)
            overlap_end = min(seg['end_ms'], w_end)
            seg_dur = seg['end_ms'] - seg['start_ms']
            if seg_dur > 0:
                ratio = (overlap_end - overlap_start) / seg_dur
                n_words = len(seg['text'].split())
                word_count += n_words * ratio
        wps = word_count / (window_ms / 1000)
        windows.append({
            'time_ms': t + window_ms // 2,
            'wps': round(wps, 2),
        })
    return windows


def compute_energy_windows(mp3_path, window_ms=5000):
    """Calcula RMS energy normalizada desde el MP3."""
    try:
        import librosa
    except ImportError:
        print(f'  librosa no instalado, saltando energy para {mp3_path.name}')
        return []

    y, sr = librosa.load(str(mp3_path), sr=16000, mono=True)
    hop = int(sr * window_ms / 1000)
    windows = []
    for i in range(0, len(y), hop):
        chunk = y[i:i + hop]
        if len(chunk) == 0:
            continue
        rms = float(np.sqrt(np.mean(chunk ** 2)))
        windows.append({
            'time_ms': int((i + hop // 2) * 1000 / sr),
            'rms': round(rms, 6),
        })

    if windows:
        rms_vals = [w['rms'] for w in windows]
        p5 = np.percentile(rms_vals, 5)
        p95 = np.percentile(rms_vals, 95)
        rng = p95 - p5 if p95 > p5 else 1.0
        for w in windows:
            w['normalized_energy'] = round(float(np.clip((w['rms'] - p5) / rng, 0, 1)), 4)
            w['is_loud'] = w['normalized_energy'] > 0.7

    return windows


print('Funciones de conversión cargadas.')

In [ ]:
# Generar artifacts/llm_{session_num}/audio.json para cada sesion
# Compatible con pipeline RAG (Notebook 02: Normalizacion, Notebook 03: Pseudo-labels)

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

# Mapear session_num -> mp3_path
mp3_by_session = {}
for mp3_path in sorted(PROJECT_ROOT.glob('recordings/*.mp3')):
    if mp3_path.stat().st_size > 1_000_000:
        num = mp3_path.stem.split('_')[0]
        mp3_by_session[num] = mp3_path

generated = 0
for session_data in all_sessions:
    session_num = session_data['session']
    segments = session_data['segments']
    duration_ms = int(session_data['duration_min'] * 60 * 1000)

    out_dir = ARTIFACTS_DIR / f'llm_{session_num}'
    out_file = out_dir / 'audio.json'

    if out_file.exists():
        print(f'[{session_num}] ya existe, saltando...')
        continue

    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n[{session_num}] Generando audio.json...')

    # Silence spans
    silence_spans = compute_silence_spans(segments, duration_ms)
    print(f'  Silencios: {len(silence_spans)}')

    # WPS windows
    wps_windows = compute_wps_windows(segments, duration_ms)
    print(f'  WPS windows: {len(wps_windows)}')

    # Energy windows (desde MP3)
    mp3_path = mp3_by_session.get(session_num)
    energy_windows = []
    if mp3_path:
        print(f'  Calculando energy desde {mp3_path.name}...')
        energy_windows = compute_energy_windows(mp3_path)
        print(f'  Energy windows: {len(energy_windows)}')

    # Generar word_spans aproximados (split de segmentos)
    word_spans = []
    for seg in segments:
        words = seg['text'].split()
        if not words:
            continue
        seg_dur = seg['end_ms'] - seg['start_ms']
        word_dur = seg_dur / len(words) if words else 0
        for j, word in enumerate(words):
            word_spans.append({
                'start_ms': int(seg['start_ms'] + j * word_dur),
                'end_ms': int(seg['start_ms'] + (j + 1) * word_dur),
                'word': word,
                'confidence': 0.9,
            })

    full_text = ' '.join(seg['text'] for seg in segments)

    audio_json = {
        'audio': {
            'status': 'completed',
            'model': session_data.get('model', 'mlx-whisper-large-v3-turbo'),
            'device': 'mlx',
            'duration_ms': duration_ms,
            'language': 'es',
            'segments': segments,
            'word_spans': word_spans,
            'full_text': full_text,
            'audio_features': {
                'energy_windows': energy_windows,
                'speech_rate_windows': wps_windows,
                'silence_spans': silence_spans,
                'speaker_turns': [],
            },
            'semantic_blocks': [],
        }
    }

    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(audio_json, f, indent=2, ensure_ascii=False)

    size_kb = out_file.stat().st_size / 1024
    print(f'  Guardado: {out_file} ({size_kb:.0f} KB)')
    generated += 1

print(f'\nGenerados: {generated} nuevos artifacts.')

---

## Resumen

In [ ]:
print("=" * 70)
print("NOTEBOOK 01: TRANSCRIPCION COMPLETADO")
print("=" * 70)
print(f"")
print(f"Modelo: mlx-community/whisper-large-v3-turbo")
print(f"Sesiones transcritas: {len(all_sessions)}")
print(f"Duracion total: {total_dur:.0f} min ({total_dur/60:.1f} horas)")
print(f"Segmentos totales: {total_segs:,}")
print(f"")
print(f"Outputs:")
print(f"  outputs/llm_sessions/*_whisper.json  (transcripciones individuales)")
print(f"  outputs/llm_sessions/all_sessions_whisper.json  (consolidado)")
print(f"  outputs/llm_sessions/transcripcion_completa.txt  (texto plano)")
print(f"  artifacts/llm_*/audio.json  (formato pipeline RAG)")
print(f"")
print(f"Proximo paso: Notebook 02 (Normalizacion de Transcripciones)")
print("=" * 70)